# Conteo por capa

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "neo4j")
import config  # noqa: E402
from neo4j import GraphDatabase

driver = GraphDatabase.driver(config.NEO4J_URI, auth=(config.NEO4J_USER, config.NEO4J_PASSWORD))


def consulta(cypher):
    with driver.session() as session:
        return pd.DataFrame([dict(fila) for fila in session.run(cypher)])


#cada nodo tiene un solo label, asi que labels(n)[0] es su capa
assert consulta("MATCH (n) WHERE size(labels(n)) <> 1 RETURN count(n) AS n").n[0] == 0

/home/ezequel/.local/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Nodos

In [2]:
nodos = consulta("""
    MATCH (n)
    RETURN labels(n)[0] AS capa, count(n) AS nodos
    ORDER BY nodos DESC
""").set_index("capa")
nodos.loc["total"] = nodos.sum()
nodos

,nodos
capa,
Drug,83867
Gene,17743
Disease,16079
Pathway,2017
total,119706


## Aristas

In [3]:
#par de capas en orden alfabetico, para que el sentido guardado no parta una relacion en dos
aristas = consulta("""
    MATCH (a)-[r]->(b)
    WITH labels(a)[0] AS la, labels(b)[0] AS lb
    RETURN CASE WHEN la <= lb THEN la ELSE lb END AS capa_1,
           CASE WHEN la <= lb THEN lb ELSE la END AS capa_2,
           count(*) AS aristas
""")
matriz = (aristas.pivot_table(index="capa_1", columns="capa_2", values="aristas", fill_value=0)
          .astype(int))
matriz

capa_2,Disease,Drug,Gene,Pathway
capa_1,,,,
Disease,16433,8496,84014,0
Drug,0,72557,131973,0
Gene,0,0,111936,42643


In [4]:
driver.close()